# Genesis Manthan — Phase 2: GRPO Training on Kaggle T4

**Runtime**: Kaggle T4 GPU (16GB VRAM)  
**Duration**: 8–10 hours per session (run 3 sessions across different days)  
**Output**: `shahansha/Manthan-1.5B` (final model) on HuggingFace Hub

**Before running**: Complete Phase 1 SFT (`02_sft_kaggle.ipynb`) and set `HF_TOKEN` secret.

**Resuming**: If the session times out, re-run from Cell 5 with `RESUME_FROM_CHECKPOINT = True`.  
The script pushes a checkpoint every 50 GRPO steps to HuggingFace Hub.

In [ ]:
# 1. INSTALL DEPENDENCIES
# Let unsloth[kaggle-new] pull in its required transformers/trl versions.
# DO NOT pin transformers or trl — unsloth 2026.x requires transformers>=4.51.3 and trl>=0.18.2.
import subprocess, sys, types, importlib.abc, importlib.machinery

# Remove Kaggle preinstalls that create resolver noise but are not used in this notebook.
# gcsfs/s3fs pin incompatible fsspec versions, and bigframes pulls optional BigQuery deps.
for pkg in ['bigframes', 'gcsfs', 's3fs']:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', pkg], capture_output=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'unsloth[kaggle-new]'], check=True)

packages = [
    'datasets>=2.20.0',
    'peft>=0.12.0',
    'accelerate>=0.33.0',
    'huggingface_hub',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade-strategy', 'only-if-needed', pkg], check=True)

# Uninstall broken optional packages that get imported transitively by Unsloth/TRL extras
for stub_pkg in ['mergekit', 'llm_blender', 'llm-blender']:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-q', '-y', stub_pkg],
                   capture_output=True)

# ---------------------------------------------------------------------------
# Stub system — uses a MetaPathFinder so Python's import machinery intercepts
# ALL dotted sub-imports (e.g. `from weave.trace import X`) at the system level.
# __getattr__ on a module is only called for attribute access, NOT for
# `import pkg.sub` — that goes through sys.meta_path instead.
# ---------------------------------------------------------------------------

# Packages whose entire import tree should be stubbed out
_STUB_ROOTS = ('mergekit', 'llm_blender', 'weave')


class _Stub:
    """Universal no-op object."""
    def __init__(self, *_args, **_kwargs): pass
    def __call__(self, *_args, **_kwargs):
        if len(_args) == 1 and callable(_args[0]) and not _kwargs:
            return _args[0]   # transparent decorator
        return _Stub()
    def __getattr__(self, _name): return _Stub()
    def __iter__(self): return iter([])
    def __len__(self): return 0
    def __contains__(self, _item): return False
    def __bool__(self): return False
    def __enter__(self): return self
    def __exit__(self, *_args): pass
    def __repr__(self): return '<Stub>'
    def __str__(self): return ''
    def __int__(self): return 0
    def __float__(self): return 0.0
    def __index__(self): return 0


class _StubModule(types.ModuleType):
    """Stub module — any attribute access returns a _Stub() instance."""
    __version__ = '0.0.0-stub'
    __file__ = '<stub>'
    __path__ = []   # makes Python treat it as a package
    __spec__ = None

    def __getattr__(self, name: str):
        stub = _Stub()
        object.__setattr__(self, name, stub)
        return stub


class _StubLoader(importlib.abc.Loader):
    """Loader that creates a _StubModule for any intercepted import."""
    def create_module(self, spec):
        # Reuse existing stub if already registered
        if spec.name in sys.modules:
            return sys.modules[spec.name]
        return _StubModule(spec.name)

    def exec_module(self, module):
        sys.modules[module.__name__] = module   # ensure it's registered


class _StubFinder(importlib.abc.MetaPathFinder):
    """
    Intercepts imports of any module under _STUB_ROOTS.
    This is called by Python's import system before it tries to find the real
    file on disk, so `import weave.trace` and `from weave.trace import X`
    both get redirected to _StubLoader without ever hitting the filesystem.
    """
    def find_spec(self, fullname, _path, _target=None):
        if any(fullname == root or fullname.startswith(root + '.') for root in _STUB_ROOTS):
            return importlib.machinery.ModuleSpec(fullname, _StubLoader())
        return None


# Remove any previously registered stub finders (idempotent re-run)
sys.meta_path = [finder for finder in sys.meta_path if not isinstance(finder, _StubFinder)]

# Flush any stale real-module cache entries for our stub roots
for module_name in list(sys.modules.keys()):
    if any(module_name == root or module_name.startswith(root + '.') for root in _STUB_ROOTS):
        del sys.modules[module_name]

# Register the finder at the FRONT of sys.meta_path so it wins over real finders
sys.meta_path.insert(0, _StubFinder())

print(f'Stub meta-path finder registered for: {", ".join(_STUB_ROOTS)}')
print('Covers all sub-imports: weave.trace, mergekit.config.*, llm_blender.blender, ...')

# Import unsloth before transformers/trl so all runtime patches are active.
import unsloth  # must be imported before trl/transformers
import importlib
print(f'unsloth: {unsloth.__version__}')
for lib in ['transformers', 'trl', 'peft']:
    try:
        mod = importlib.import_module(lib)
        print(f'{lib}: {mod.__version__}')
    except Exception as e:
        print(f'{lib}: FAILED — {e}')

print('✅ Dependencies installed')


In [ ]:
# 2. AUTHENTICATE
import unsloth  # must be imported before trl/transformers to apply all patches
import os
from huggingface_hub import login, HfApi

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    hf_token = os.environ.get('HF_TOKEN')

if not hf_token:
    raise ValueError('HF_TOKEN not found.')
login(token=hf_token, add_to_git_credential=False)

# Verify token + get exact username casing (avoids 403 on repo creation)
api = HfApi()
whoami = api.whoami(token=hf_token)
hf_username = whoami['name']
token_role = whoami.get('auth', {}).get('accessToken', {}).get('role', 'unknown')
print(f'HF user: {hf_username} | token role: {token_role}')

if token_role not in ('write', 'admin'):
    raise RuntimeError(
        f"Token role is '{token_role}' — write access required.\n"
        "Go to https://huggingface.co/settings/tokens → New token → Role: Write\n"
        "Then update HF_TOKEN in Kaggle Secrets."
    )
print('✅ HuggingFace authenticated — write access confirmed')


In [ ]:
# 3. CONFIGURATION
# Adjust these for your session

# Use exact username casing from whoami (set in Cell 2) to avoid 403 on Hub push
SFT_MODEL_ID = f'{hf_username}/Manthan-1.5B-sft'   # Phase 1 output
HUB_MODEL_ID = f'{hf_username}/Manthan-1.5B'        # Phase 2 output (final model)
MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 512
ROLLOUT_ANSWER_TOKENS = 96  # Follow-up answer budget after tool observation
RESUME_FROM_CHECKPOINT = False  # Set True if resuming a timed-out session

# GRPO hyperparameters (T4-optimized)
NUM_GENERATIONS = 4      # Samples per prompt for relative advantage
PER_DEVICE_BATCH = 1     # Must be 1 for GRPO on T4
GRAD_ACCUM = 8           # Effective batch size = 8
LEARNING_RATE = 5e-6
MAX_STEPS = 450          # ~8–10 hours on T4 for ~3 sessions
SAVE_STEPS = 50          # Push to Hub every 50 steps

print(f'Config:')
print(f'  Base model: {SFT_MODEL_ID}')
print(f'  Output: {HUB_MODEL_ID}')
print(f'  Max steps: {MAX_STEPS}')
print(f'  Rollout answer tokens: {ROLLOUT_ANSWER_TOKENS}')
print(f'  Resuming: {RESUME_FROM_CHECKPOINT}')


In [ ]:
# 4. LOAD MODEL
import torch
from unsloth import FastLanguageModel

print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,  # CRITICAL: float16 only for T4
    load_in_4bit=True,
)

if hasattr(model, 'generation_config') and model.generation_config is not None:
    model.generation_config.max_length = None

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'✅ Model loaded: {SFT_MODEL_ID}')
model.print_trainable_parameters()

In [ ]:
# 5. DEFINE REWARD FUNCTIONS
import json, re, subprocess, tempfile, os, textwrap
import torch

TOOL_CALL_RE = re.compile(r'<tool_call>(.*?)(?:</tool_call>|<tool_response>|<final_answer>|$)', re.DOTALL)
FINAL_ANSWER_RE = re.compile(r'<final_answer>(.*?)</final_answer>', re.DOTALL)

def execute_code_sandbox(code: str, timeout: int = 10) -> dict:
    """Execute code in a sandboxed subprocess. Returns {success, result, error}."""
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(textwrap.dedent(code))
        tmp_path = f.name
    try:
        result = subprocess.run(
            ['python', tmp_path],
            capture_output=True, text=True,
            timeout=timeout,
            env={**os.environ, 'PYTHONDONTWRITEBYTECODE': '1'},
        )
        stdout = result.stdout[:10240]  # 10KB cap
        if result.returncode == 0:
            return {'success': True, 'result': stdout.strip(), 'error': None}
        return {'success': False, 'result': stdout.strip(), 'error': result.stderr[:1024]}
    except subprocess.TimeoutExpired:
        return {'success': False, 'result': '', 'error': 'timeout'}
    except Exception as e:
        return {'success': False, 'result': '', 'error': str(e)}
    finally:
        os.unlink(tmp_path)


def _as_text(completion) -> str:
    """Normalise completions that may arrive as strings or chat messages."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for msg in completion:
            if isinstance(msg, dict):
                parts.append(msg.get('content', ''))
            else:
                parts.append(str(msg))
        return '\n'.join(parts)
    return str(completion)


def _prompt_to_chatml(prompt) -> str:
    """Convert a prompt payload into the ChatML format used during SFT."""
    if isinstance(prompt, str):
        return prompt
    if isinstance(prompt, list):
        parts = []
        for msg in prompt:
            if not isinstance(msg, dict):
                continue
            role = str(msg.get('role', 'user'))
            content = str(msg.get('content', '')).strip()
            parts.append(f'<|im_start|>{role}\n{content}<|im_end|>')
        return '\n'.join(parts)
    return str(prompt)


def _extract_first_tool_call_payload(completion_text: str) -> str | None:
    match = TOOL_CALL_RE.search(completion_text)
    if not match:
        return None
    return match.group(1).strip()


def _build_tool_response_block(sandbox_result: dict) -> str:
    payload = {
        'result': sandbox_result.get('result', ''),
        'success': bool(sandbox_result.get('success', False)),
    }
    error = str(sandbox_result.get('error', '') or '').strip()
    if error:
        payload['error'] = error
    return f'<tool_response>{json.dumps(payload, ensure_ascii=True)}</tool_response>'


def _build_rollout_prompt(prompt, tool_call_block: str, sandbox_result: dict) -> str:
    prompt_text = _prompt_to_chatml(prompt).strip()
    tool_response_block = _build_tool_response_block(sandbox_result)
    return (
        f'{prompt_text}\n'
        f'<|im_start|>assistant\n{tool_call_block}<|im_end|>\n'
        f'<|im_start|>tool\n{tool_response_block}<|im_end|>\n'
        '<|im_start|>assistant\n'
    )


def _generate_followup_answer(rollout_prompt: str, max_new_tokens: int = ROLLOUT_ANSWER_TOKENS) -> str:
    device = next(model.parameters()).device
    inputs = tokenizer(rollout_prompt, return_tensors='pt').to(device)
    was_training = model.training
    pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
    if hasattr(model, 'generation_config') and model.generation_config is not None:
        model.generation_config.max_length = None
    try:
        model.eval()
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=pad_token_id,
            )
    finally:
        if was_training:
            model.train()

    generated = outputs[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=False)
    return text.split('<|im_end|>', 1)[0].strip()


def _extract_final_answer(completion: str) -> str | None:
    m = FINAL_ANSWER_RE.search(completion)
    return m.group(1).strip() if m else None


def tool_execution_reward(completion: str, sandbox_result: dict | None = None) -> float:
    payload = _extract_first_tool_call_payload(completion)
    if payload is None:
        return 0.0
    try:
        data = json.loads(payload)
    except json.JSONDecodeError:
        return 0.0
    if isinstance(data, list):
        if not data:
            return 0.0
        data = data[0]
    if not isinstance(data, dict):
        return 0.0
    code = data.get('arguments', {}).get('code', '')
    if not code or len(code.strip()) < 5:
        return 0.25
    if sandbox_result is None:
        return 0.5
    if not sandbox_result.get('success'):
        return 0.5
    return 1.0 if str(sandbox_result.get('result', '')).strip() else 0.75


def answer_correctness_reward(completion: str, ground_truth: str) -> float:
    answer = _extract_final_answer(completion)
    if answer is None:
        return 0.0
    if answer.strip().lower() == ground_truth.strip().lower():
        return 1.0
    try:
        a, b = float(answer.strip()), float(ground_truth.strip())
        rel_err = abs(a - b) / max(abs(b), 1e-9)
        if rel_err <= 0.001:
            return 0.9
        if rel_err <= 0.01:
            return 0.5
    except ValueError:
        pass
    return 0.0


def format_reward(completion: str) -> float:
    return 0.1 if re.search(r'<tool_call>', completion) else 0.0


print('✅ Reward functions defined')

# Quick sanity check
test_completion = '<tool_call>{"name": "python_repl", "arguments": {"code": "print(42)"}}</tool_call>'
test_result = execute_code_sandbox('print(42)')
test_rollout = '\n'.join([
    test_completion,
    _build_tool_response_block(test_result),
    '<final_answer>42</final_answer>',
])
print(f'Sandbox test: {test_result}')
print(f'Tool exec reward: {tool_execution_reward(test_completion, test_result):.2f}')
print(f'Answer reward: {answer_correctness_reward(test_rollout, "42"):.2f}')


In [ ]:
# 6. LOAD REWARD DATASET
from datasets import load_dataset as hf_load_dataset

# Use GSM8K train split as GRPO training problems
gsm8k = hf_load_dataset('gsm8k', 'main', split='train')

def extract_gsm8k_answer(solution: str) -> str:
    m = re.search(r'####\s*([\d,.-]+)', solution)
    if m:
        return m.group(1).replace(',', '').strip()
    return solution.strip().split('\n')[-1]

SYSTEM_PROMPT = (
    'You are Genesis Manthan. Never reason verbally. On the first assistant turn, emit '
    'exactly one <tool_call> for python_repl and stop. After a tool result is provided, '
    'reply with <final_answer>.'
)

grpo_dataset = []

for item in gsm8k.select(range(500)):  # Use 500 problems for GRPO
    problem = item['question']
    answer = extract_gsm8k_answer(item['answer'])
    grpo_dataset.append({
        'prompt': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': problem},
        ],
        # GRPOTrainer passes extra dataset columns as kwargs to the reward function.
        'answer': answer,
    })

from datasets import Dataset
grpo_hf_dataset = Dataset.from_list(grpo_dataset)

print(f'GRPO dataset: {len(grpo_hf_dataset)} problems')
print(f'Sample problem: {grpo_hf_dataset[0]["prompt"][1]["content"][:100]}...')
print(f'Sample answer: {grpo_hf_dataset[0]["answer"]}')


def reward_fn(completions, prompts=None, answer=None, **kwargs) -> list[float]:
    rewards = []
    for i, completion in enumerate(completions):
        text = _as_text(completion)
        prompt = prompts[i] if prompts else ''
        gt = (answer[i] if answer else '') or ''

        sandbox_result = None
        rollout_completion = text
        tool_call_payload = _extract_first_tool_call_payload(text)

        if tool_call_payload:
            try:
                parsed = json.loads(tool_call_payload)
                if isinstance(parsed, list):
                    if parsed:
                        parsed = parsed[0]
                    else:
                        parsed = None
                if isinstance(parsed, dict):
                    code = parsed.get('arguments', {}).get('code', '')
                    if code and len(code.strip()) >= 5:
                        sandbox_result = execute_code_sandbox(code)
                        tool_call_block = f'<tool_call>{tool_call_payload}</tool_call>'
                        rollout_prompt = _build_rollout_prompt(prompt, tool_call_block, sandbox_result)
                        followup_text = _generate_followup_answer(rollout_prompt)
                        rollout_completion = '\n'.join([
                            tool_call_block,
                            _build_tool_response_block(sandbox_result),
                            followup_text,
                        ]).strip()
            except (json.JSONDecodeError, AttributeError, TypeError):
                pass

        r1 = tool_execution_reward(text, sandbox_result)
        r2 = answer_correctness_reward(rollout_completion, gt)
        r3 = format_reward(text)
        combined = min(1.0, 0.5 * r1 + 0.4 * r2 + 0.1 * r3)
        rewards.append(combined)
    return rewards

print('✅ Reward dataset and rollout-aware function ready')


In [ ]:
# 7. RUN GRPO TRAINING
import unsloth  # re-affirm import order before trl
from trl import GRPOTrainer, GRPOConfig

training_args = GRPOConfig(
    output_dir='/kaggle/working/manthan-grpo',
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_steps=20,
    fp16=True,
    bf16=False,           # CRITICAL: T4 does not support bfloat16
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,  # renamed from max_new_tokens in trl>=0.15
    temperature=0.9,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID,
    hub_token=hf_token,   # explicit — avoids falling back to a cached read-only token
    hub_strategy='every_save',
    report_to='none',
    seed=42,
    use_vllm=False,       # vLLM not available on T4 with 4-bit
    # Workaround: newer transformers defaults average_tokens_across_devices=True,
    # which converts Unsloth's fused loss tensor to int, breaking .mean() (unsloth#3769)
    average_tokens_across_devices=False,
)

# Workaround: trl GRPOTrainer accesses model.warnings_issued but unsloth-patched
# models don't always have this attribute. Set it on both the outer model and the
# inner base model to be safe.
if not hasattr(model, 'warnings_issued'):
    model.warnings_issued = {}
if hasattr(model, 'model') and not hasattr(model.model, 'warnings_issued'):
    model.model.warnings_issued = {}

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=grpo_hf_dataset,
    reward_funcs=[reward_fn],
)

print(f'Starting GRPO training for {MAX_STEPS} steps...')
print(f'Checkpoints pushed every {SAVE_STEPS} steps to: {HUB_MODEL_ID}')
print('Expected duration: 8–10 hours')

if RESUME_FROM_CHECKPOINT:
    trainer.train(resume_from_checkpoint='/kaggle/working/manthan-grpo')
else:
    trainer.train()


In [ ]:
# 8. SAVE AND PUSH FINAL MODEL
model.save_pretrained('/kaggle/working/manthan-grpo-final')
tokenizer.save_pretrained('/kaggle/working/manthan-grpo-final')

# Push merged model (LoRA merged into base weights)
model.push_to_hub_merged(
    HUB_MODEL_ID,
    tokenizer=tokenizer,
    save_method='merged_16bit',
)
print(f'✅ Final model pushed to: https://huggingface.co/{HUB_MODEL_ID}')

In [ ]:
# 9. EXPORT GGUF Q4_K_M (do this at session end, saves separate upload)
model.push_to_hub_gguf(
    HUB_MODEL_ID,
    tokenizer=tokenizer,
    quantization_method=['q4_k_m', 'q8_0'],
)
print('✅ GGUF quantized models pushed to Hub')

In [ ]:
# 10. QUICK BENCHMARK: Tool parsability
import re
FastLanguageModel.for_inference(model)

test_problems = [
    'What is 17 × 23?',
    'Find the sum of all prime numbers below 50.',
    'What is the area of a circle with radius 7?',
    'Calculate 15% of 340.',
    'How many seconds are in a week?',
]

tool_calls_generated = 0
parsable = 0

for problem in test_problems:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': problem},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to('cuda')
    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    has_tool_call = '<tool_call>' in response
    tool_calls_generated += int(has_tool_call)

    if has_tool_call:
        m = re.search(r'<tool_call>(.*?)</tool_call>', response, re.DOTALL)
        if m:
            try:
                json.loads(m.group(1).strip())
                parsable += 1
            except json.JSONDecodeError:
                pass
    print(f'  Problem: {problem[:40]} | Tool call: {has_tool_call}')

print(f'\nTool call rate: {tool_calls_generated}/{len(test_problems)} = {tool_calls_generated/len(test_problems):.0%}')
print(f'Parsability: {parsable}/{tool_calls_generated} = {parsable/max(tool_calls_generated,1):.0%}')

## Session Complete

Model is now at `shahansha/Manthan-1.5B` on HuggingFace Hub.

**If you timed out before reaching target steps:**
1. Start a new Kaggle session
2. Re-run all cells
3. In Cell 3, set `RESUME_FROM_CHECKPOINT = True`
4. The trainer will resume from the last Hub checkpoint automatically

**Target steps**: 300–450 minimum for meaningful GRPO improvement.

**Next**: Evaluate with `src/eval/benchmark_gsm8k.py` and `src/eval/benchmark_mbpp.py`.